# 16 · Reduce the effect sizes to what carries signal

Notebook 15 fits every knockout against every response gene, which is far more
than the module clustering can use: most of that matrix is noise. This step
corrects it for multiple testing and keeps the rows and columns that carry
signal.

A knockout is kept when it moves more than `par_significant_target_cutoff`
genes at `par_effect_fdr_cutoff`. A gene is kept when more than
`par_significant_gene_cutoff` knockouts move it. The surviving submatrix is
what notebook 17 clusters into gene and guide modules.

**Reads** `par_effect_coefs_file` and `par_effect_pvals_file`.
**Writes** `par_effect_fdr_file`, `par_effect_adjusted_pvals_file` and
`par_selected_coef_matrix_recomputed_file`.

The control fits, if present, give the null: the same model run over control
guides, which are not perturbations. Comparing how many genes a control
"affects" against how many a knockout affects is what makes the cutoff a
judgement rather than a guess.

## Setup

In [ ]:
from libraries import *
from parameters import *
from pathlib import Path
import statsmodels.stats.multitest as smm

os.chdir(projectDir)

## Correct for multiple testing

Corrected within each gene, across knockouts — the question being asked of each
column is which knockouts moved this gene.

In [ ]:
coefs = pd.read_csv(par_effect_coefs_file, index_col=0)
pvals = pd.read_csv(par_effect_pvals_file, index_col=0)
# the fitted rows include the intercept, the quality covariates and the
# random-effect variance; only the knockouts are effects to be corrected
not_effects = {"Intercept", "n_genes", "mt_frac", "Group Var"}
coefs = coefs.loc[[r for r in coefs.index if r not in not_effects]]
pvals = pvals.loc[[r for r in pvals.index if r not in not_effects]]
print(f"input: {coefs.shape[0]} knockouts x {coefs.shape[1]} genes")

fdr = pvals.apply(lambda column: smm.multipletests(column.fillna(1), method="fdr_bh")[1],
                  axis=0, result_type="broadcast")
fdr.to_csv(par_effect_fdr_file)
print(f"written: {par_effect_fdr_file}")

## How many genes does each knockout move?

`affected` counts, for each knockout, the genes it moves at the chosen FDR. The
same count over the control fits is the null: a control guide is not a
perturbation, so whatever it appears to move is the floor.

In [ ]:
affected = (fdr < par_effect_fdr_cutoff).sum(axis=1)

control_affected = None
if Path(par_effect_control_pvals_input).exists():
    control_pvals = pd.read_csv(par_effect_control_pvals_input, index_col=0)
    control_fdr = control_pvals.apply(
        lambda column: smm.multipletests(column.fillna(1), method="fdr_bh")[1],
        axis=0, result_type="broadcast")
    control_affected = (control_fdr < par_effect_fdr_cutoff).sum(axis=1)
    print(f"control guides: {len(control_affected)}")
    print(f"  genes moved, median {control_affected.median():.0f}, "
          f"90th percentile {control_affected.quantile(0.9):.0f}, max {control_affected.max()}")
else:
    print(f"control fits not found at {par_effect_control_pvals_input}; skipping the null")

print(f"knockouts: {len(affected)}")
print(f"  genes moved, median {affected.median():.0f}, "
      f"90th percentile {affected.quantile(0.9):.0f}, max {affected.max()}")
print(f"\ncutoff in use: more than {par_significant_target_cutoff} genes")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
if control_affected is not None:
    axes[0].hist([control_affected, affected], bins=40, label=["control", "knockout"],
                 density=True)
    axes[0].legend()
else:
    axes[0].hist(affected, bins=40)
axes[0].axvline(par_significant_target_cutoff, color="red", linestyle="--")
axes[0].set_xlabel("genes moved per knockout"); axes[0].set_ylabel("density")

moved_by = (fdr < par_effect_fdr_cutoff).sum(axis=0)
axes[1].hist(moved_by, bins=40)
axes[1].axvline(par_significant_gene_cutoff, color="red", linestyle="--")
axes[1].set_xlabel("knockouts moving a gene"); axes[1].set_ylabel("genes")
plt.tight_layout(); plt.show()

## Reduce and write

In [ ]:
keep_knockouts = affected > par_significant_target_cutoff
keep_genes = moved_by > par_significant_gene_cutoff

print(f"knockouts kept: {int(keep_knockouts.sum())} of {len(keep_knockouts)}")
print(f"genes kept    : {int(keep_genes.sum())} of {len(keep_genes)}")

significant = coefs.loc[keep_knockouts, keep_genes]
adjusted = fdr.loc[keep_knockouts, keep_genes]

Path(par_selected_coef_matrix_recomputed_file).parent.mkdir(parents=True, exist_ok=True)
significant.to_csv(par_selected_coef_matrix_recomputed_file)
adjusted.to_csv(par_effect_adjusted_pvals_file)

print(f"\nwritten: {par_selected_coef_matrix_recomputed_file}  {significant.shape}")
print(f"the matrix used downstream is {par_selected_coef_matrix_file}, left unchanged")